In [8]:
import networkx as nx
import numpy as np
from scipy.sparse import csc_matrix, diags

# 1. Caricamento Dati
mappatura_url = {}
lista_edges = []
with open('hollins.dat', 'r', encoding='utf-8') as f:
    intestazione = f.readline().split()
    numero_nodi = int(intestazione[0])
    numero_archi = int(intestazione[1])

    for i in range(numero_nodi):
        linea = f.readline().strip().split(' ', 1)
        node_id = int(linea[0])
        mappatura_url[node_id] = linea[1]

    for line in f:
        sorgente, destinazione = map(int, line.strip().split())
        lista_edges.append((sorgente, destinazione))

# Creazione Grafo
G = nx.DiGraph()
G.add_nodes_from(range(1, numero_nodi + 1))
G.add_edges_from(lista_edges)

# IMPORTANTE: Creiamo la lista ordinata dei nodi per garantire che l'indice i sia sempre ID i+1
nodi_ordinati = list(range(1, numero_nodi + 1))
N = numero_nodi


In [9]:
# Matrice di adiacenza con ordine prefissato
# nodelist garantisce che la riga 0 sia il nodo 1, riga 1 il nodo 2, etc.
A_nx = nx.adjacency_matrix(G, nodelist=nodi_ordinati)

# Calcolo Out-Degree
out_degrees = np.array([G.out_degree(n) for n in nodi_ordinati], dtype=float)

# Creazione Matrice Link (A_initial) - Colonna Stocastica
# Gestiamo i gradi zero per evitare divisioni per zero
with np.errstate(divide='ignore'):
    inv_out_degrees = 1.0 / out_degrees
inv_out_degrees[np.isinf(inv_out_degrees)] = 0.0

D_inv = diags(inv_out_degrees)
# A_initial ha colonne che sommano a 1 (tranne i dangling nodes che hanno colonne di 0)
A_initial = A_nx.T.dot(D_inv)

# Vettore h per i dangling nodes (1 se il nodo è dangling, 0 altrimenti)
h = (out_degrees == 0).astype(float).reshape(N, 1)

In [10]:
def PowerMethod_Stable(A_zero_cols, N, m, h, epsilon=1e-9, maxiter=500):
    d = 1.0 - m
    s = np.full((N, 1), 1.0 / N)  # Vettore di teletrasporto
    xk = s.copy() 

    for k in range(maxiter):
        xk_prev = xk.copy()

        # 1. Calcola la massa persa dai dangling nodes
        mass_lost = h.T.dot(xk_prev) # Scalare: quanta probabilità è "finita nel vuoto"

        # 2. Iterazione PageRank (Formula standard)
        # x = (1-m) * (A*x + s*mass_lost) + m*s
        xk_new = d * (A_zero_cols.dot(xk_prev) + s * mass_lost) + m * s

        # 3. Verifica Convergenza (Norma L1 è più precisa per probabilità)
        if np.linalg.norm(xk_new - xk_prev, ord=1) < epsilon:
            print(f"Convergenza raggiunta dopo {k+1} iterazioni.")
            return xk_new
        xk = xk_new

    return xk

# Esecuzione
m = 0.15
pagerank_vettore = PowerMethod_Stable(A_initial, N, m, h)

Convergenza raggiunta dopo 97 iterazioni.


In [11]:
def PowerMethod(A, N, m=0.15, relTol=1e-8, maxiter=500):
    # Initialize s (vector of lenght N with all values = 1/N)
    s = np.full((N, 1), 1.0 / N)

    # Initialize and normalize xk (the starting vector)
    xk = s.copy()
    xk = xk / np.linalg.norm(xk, ord=1) 
    print(sum(xk))
    k = 0 
    
    print(f"Inizio iterazioni...")

    while k < maxiter:
        
        # Compute the new vector x_tilde_k+1, this is a vector NOT NORMALIZED
        # This computation exploits the sparsity of the matrix A by computing the @ product of A @ xk and it deals with 
        # dangling nodes by using the m = 0.15

        xk_new_tilde = (1 - m) * (A @ xk) + (m * s) 
        print(xk_new_tilde.sum())
        # Normalize the xk_tilde
        norm_val = np.linalg.norm(xk_new_tilde, ord=1)
        xk_new = xk_new_tilde / norm_val
        
        #Check the convergence

        diff = np.linalg.norm(xk_new - xk)
        if diff < relTol:

            print(f"Convergenza raggiunta all'iterazione {k+1}")
            xk = xk_new
            break
        
        xk = xk_new
        k += 1
        
    return xk

In [12]:
m = 0.15
pagerank_vettore = PowerMethod(A_initial, N, m)
print(PowerMethod(A_initial, N, m))
#print(nx.pagerank(G, alpha=0.85))

[1.]
Inizio iterazioni...
0.549126746506986
0.7888039952704626
0.8464206858325755
0.8656651737461492
0.8714775929234857
0.8755032812907807
0.8779629004082401
0.8802057264023013
0.8813721472580498
0.8823555915978523
0.8830811708477958
0.8837007449336328
0.8842610114009339
0.8847169340530397
0.8851722967840299
0.885537113724545
0.8859064656595728
0.8862098916762795
0.8865206633663396
0.8867752450033003
0.8870374430369616
0.8872525363446567
0.8874726906114327
0.8876544228347965
0.8878396923678961
0.8879928321943666
0.8881484575235876
0.8882773486163031
0.8884077626599225
0.888516073099455
0.8886252663123013
0.8887161417954268
0.8888074708627893
0.8888836459088706
0.8889599654556155
0.8890237699588991
0.8890875167483758
0.8891409294459094
0.8891941567727633
0.8892388558701458
0.8892832911112973
0.8893206917651653
0.8893577860837343
0.8893890785899992
0.8894200470371907
0.889446230906177
0.8894720893865884
0.8894940022538975
0.8895155992956401
0.8895339422769786
0.8895519858372036
0.8895673

In [13]:
pagerank_vettore

array([[2.80419481e-05],
       [1.76690911e-02],
       [5.94542075e-05],
       ...,
       [3.43596944e-05],
       [3.43596944e-05],
       [1.45427220e-04]], shape=(6012, 1))

In [15]:
# Trasformiamo il vettore in una lista piatta di score
pagerank_scores = pagerank_vettore.flatten()

classifica_completa = []
for i in range(N):
    node_id = nodi_ordinati[i] # Questo garantisce la corrispondenza corretta
    score = pagerank_scores[i]
    url = mappatura_url.get(node_id, "URL non trovato")
    classifica_completa.append((score, node_id, url))

# Ordinamento decrescente
classifica_completa.sort(key=lambda item: item[0], reverse=True)

print(f"\n🥇 RANKING REFACTORED (m={m})")
print("-" * 85)
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (score, node_id, url) in enumerate(classifica_completa[:10], start=1):
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")

# --- RISULTATI NETWORKX (FIX) ---
print("\n--- RISULTATI NETWORKX ---")
# Calcolo PageRank con NetworkX
nx_scores_dict = nx.pagerank(G, alpha=0.85)

# 1. Convertiamo il dizionario in una lista di tuple (ID, Score) e ordiniamo per Score decrescente
nx_ranking_sorted = sorted(nx_scores_dict.items(), key=lambda item: item[1], reverse=True)

# 2. Stampa formattata
print(f"{'Rank':<5} {'Score':<15} {'ID':<8} {'URL'}")
print("-" * 85)

for rank, (node_id, score) in enumerate(nx_ranking_sorted[:10], start=1):
    url = mappatura_url.get(node_id, "URL non trovato")
    print(f"{rank:<5} {score:.10f} {node_id:<8} {url}")


🥇 RANKING REFACTORED (m=0.15)
-------------------------------------------------------------------------------------
Rank  Score           ID       URL
-------------------------------------------------------------------------------------
1     0.0176690911 2        http://www.hollins.edu/
2     0.0102379814 37       http://www.hollins.edu/admissions/visit/visit.htm
3     0.0094571741 38       http://www.hollins.edu/about/about_tour.htm
4     0.0090694738 61       http://www.hollins.edu/htdig/index.html
5     0.0088566283 52       http://www.hollins.edu/admissions/info-request/info-request.cfm
6     0.0081205088 4023     http://www1.hollins.edu/faculty/saloweyca/clas%20395/Sculpture/sld001.htm
7     0.0078049849 43       http://www.hollins.edu/admissions/apply/apply.htm
8     0.0069343822 27       http://www.hollins.edu/admissions/admissions.htm
9     0.0069235833 3227     http://www1.hollins.edu/faculty/saloweyca/clas%20395/Sculpture/index.htm
10    0.0063185713 5254     http://www1.ho